In [ ]:
import sys

sys.path.insert(0, "/home/hgf_hmgu/hgf_gib4562/tdmpc2/tdmpc2")

from envs import make_env
from recording_tdmpc2 import RecordingTDMPC2


def setup_agent(cfg):
    """
    Initialize environment and agent.
    
    Args:
        cfg: Configuration object
        
    Returns:
        env: Environment instance
        agent: RecordingTDMPC2 agent instance
    """
    # Initialize environment
    env = make_env(cfg)

    # Initialize recording agent
    agent = RecordingTDMPC2(cfg, planning_recorder=None)

    # Load checkpoint
    if cfg.checkpoint and cfg.checkpoint != '???':
        agent.load(cfg.checkpoint)
    else:
        raise ValueError(
            "Must provide a checkpoint path via checkpoint=path/to/checkpoint.pt"
        )

    return env, agent


In [ ]:
# This cell has been made runnable in a notebook (removes __main__ guard, no sys.path hacks for running locally).
# If using relative imports within a project, you may need to adjust paths or use %run on utility code cells first.

import torch
from pathlib import Path
from datetime import datetime

import sys

#sys.path.insert(0, "/home/hgf_hmgu/hgf_gib4562/tdmpc2/tdmpc2")

sys.path.append("/home/hgf_hmgu/hgf_gib4562/tdmpc2/sweeps")

try:
    from hydra import initialize, compose
    from omegaconf import OmegaConf
    from utils import run_episode_with_recording
    from common.episode_data_recorder import EpisodeDataRecorder
    from common.planning_data_recorder import PlanningDataRecorder
    from common.seed import set_seed
    from common.parser import parse_cfg
except ImportError as e:
    print(
        "Some imports failed. Make sure all dependencies are available and in the notebook path."
    )
    raise e


In [ ]:
# Minimal configuration - just one episode
CKPT_PATH = 'ckpts/cartpole-swingup-3.pt'
override_cfg = dict(
    task='cartpole-swingup',
    checkpoint=CKPT_PATH,
    obs='state',
    seed=1,
    compile=False,
    mpc=True,
    multitask=False,
    model_size=5,
    save_video=False,
    record_planning=False,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Base directory for saving results
BASE_SAVE_DIR = 'logs/exploration/simple_debug_sweep'

# Run everything as normal, but within the notebook!
print("=" * 80)
print("SIMPLE DEBUG SWEEP - NO INTERVENTIONS (notebook cell)")
print("=" * 80)
print(f"Task: {override_cfg['task']}")
print(f"Checkpoint: {override_cfg['checkpoint']}")
print(f"Device: {DEVICE}")
print(f"Save directory: {BASE_SAVE_DIR}")
print("=" * 80)

# Initialize Hydra config
with initialize(config_path="../tdmpc2/tdmpc2", version_base=None):
    cfg = compose(config_name="config")
    OmegaConf.set_struct(cfg, False)
    cfg = OmegaConf.merge(cfg, override_cfg)

# Parse config
cfg = parse_cfg(cfg)

# Set seed
set_seed(cfg.seed)

# Initialize environment and agent
print("\nInitializing environment and agent...")
env, agent = setup_agent(cfg)

print(f"Environment: {env}")
print(f"Agent: {agent}")
print(f"Agent device: {getattr(agent, '_device', torch.device('cpu'))}")

# Create save directory
time = datetime.now().strftime("%Y%m%d_%H%M%S")
episode_dir = Path(BASE_SAVE_DIR) / time
activations_dir = episode_dir / 'activations'
activations_dir.mkdir(parents=True, exist_ok=True)

# Create episode recorder
episode_recorder = EpisodeDataRecorder(cfg, save_dir=str(activations_dir))
planning_recorder = PlanningDataRecorder(save_dir=str(episode_dir / 'planning'),
                                         record_rollouts=True)
print("\nRunning episode...")
print("-" * 80)

# Run episode - no interventions, just basic recording
run_episode_with_recording(env=env,
                           agent=agent,
                           episode_recorder=episode_recorder,
                           planning_recorder=planning_recorder,
                           save_video=cfg.save_video,
                           eval_mode=True,
                           task=None,
                           task_name=cfg.task,
                           episode_dir=str(episode_dir))

# Save episode data
metadata = {
    'seed': cfg.seed,
    'checkpoint': str(cfg.checkpoint),
    'task': cfg.task,
    'note': 'Simple debug run - no interventions',
}
# Uncomment if you want to save
# episode_recorder.save_episode(metadata=metadata)

print("-" * 80)
print(f"\n✓ Episode completed successfully!")
print(f"✓ Data saved to: {activations_dir}")
print("\n" + "=" * 80)
print("DEBUG SWEEP COMPLETE!")
print("=" * 80)
